# How many replicates does a check need?

Reads the ten replicates of one configuration written by

```bash
python experiments/exploration-replicate-variability/run.py
```

`replication-reporting`, ten examples from ten distinct documents, ten
identical runs. The note is
[`thinking/experiments/exploration-replicate-variability.md`](../../thinking/experiments/exploration-replicate-variability.md)
— read it first: it records that this probe aims deliberately at the noisiest
check, so the number here is closer to an upper bound than a typical value.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc import cli

RUNS = Path("../../experiments/runs/exploration-replicate-variability").resolve()
CHECKLIST, CHECK, MODEL = "fig-checklist-exp01", "replication-reporting", "claude-sonnet-5"
FULL_BENCHMARK = 38   # examples exp-01 will run; this probe uses ten of them

reps = sorted((RUNS / "pinned").glob("rep-*"))
assert reps, f"no replicates under {RUNS}; run the probe first"
[p.name for p in reps]

2026-09-20 22:11:08 - INFO - ✅ Anthropic API provider configured


['rep-00',
 'rep-01',
 'rep-02',
 'rep-03',
 'rep-04',
 'rep-05',
 'rep-06',
 'rep-07',
 'rep-08',
 'rep-09']

## One score per example per replicate

`score_check` takes one `rep-NN/`. Ten calls give a 10 × 10 grid: the same
examples, measured ten times.

In [2]:
rows = []
for rep in reps:
    index = int(rep.name.split("-")[1])
    result = cli.score_check(CHECKLIST, CHECK, rep, model=MODEL, save=False)
    for record in result["agentic"]["flat"]:
        props = (record["analysis"].get("by_property") or {})
        # `mean_score` is the layer-2 rollup over instances that are applicable
        # and correctly so -- and it is **0.0 when a property has no eligible
        # instances at all**, which means "nothing to score", not "scored
        # zero". Averaging those zeros across properties both depresses the
        # score and, worse for this probe, injects variance whenever a panel
        # is classified applicable in one replicate and not in another.
        #
        # So weight each property by how many instances it actually scored:
        # a property with none contributes nothing, and every scored instance
        # carries equal weight.
        values, weights = [], []
        for prop, summary in props.items():
            mean = summary.get("mean_score")
            if mean is None:
                continue
            assert summary.get("layer1_counts") is not None, (
                f"{prop} is unprofiled; mean_score then covers all instances "
                "and this weighting needs rethinking"
            )
            eligible = sum((summary.get("layer2_counts") or {}).values())
            if eligible:
                values.append(mean)
                weights.append(eligible)
        rows.append({
            "replicate": index,
            "example": record.get("doc_id"),
            "score": float(np.average(values, weights=weights)) if values else np.nan,
        })

grid = pd.DataFrame(rows).pivot(index="example", columns="replicate", values="score")
print(f"{grid.shape[0]} examples x {grid.shape[1]} replicates")
grid.round(3)

2026-09-20 22:11:09 - WARNING - No prediction for 28 of 38 benchmark example(s) of replication-reporting; scoring the remaining 10
2026-09-20 22:11:09 - INFO - Analyzing results with FlatEvaluator (replication-reporting)
Analyzing: 0 example [00:00, ? example/s]2026-09-20 22:11:12 - INFO - 📱 CPU device selected
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Analyzing: 10 example [00:04,  2.18 example/s]

10 examples x 10 replicates


replicate,0,1,2,3,4,5,6,7,8,9
example,,,,,,,,,,
10.1038_emboj.2009.312,0.729,0.837,0.000,0.806,0.831,0.828,0.830,0.772,0.815,0.837
10.1038_emboj.2009.340,0.917,0.914,0.916,0.913,0.912,0.912,0.913,0.921,0.917,0.920
10.1038_embor.2009.217,0.881,0.976,0.974,0.978,0.972,0.879,0.910,0.968,0.970,0.982
10.1038_embor.2009.233,0.927,0.923,0.891,0.900,0.886,0.890,0.917,0.889,0.890,0.933
10.1038_s44318-025-00409-0,0.856,0.928,0.867,0.898,0.843,0.917,0.913,0.915,0.914,0.928
10.1038_s44318-026-00715-1,0.968,0.986,0.974,0.970,0.982,0.984,0.986,0.988,0.984,0.984
10.1038_s44319-025-00438-0,0.951,0.848,0.947,0.964,0.963,0.943,0.953,0.950,0.939,0.925
10.1038_s44319-025-00631-1,0.904,0.917,0.905,0.905,0.908,0.899,0.897,0.898,0.921,0.893
10.1038_s44320-025-00092-7,0.880,0.919,0.902,0.879,0.907,0.905,0.913,0.888,0.939,0.899


## The decomposition

The quantity exp-01 needs is the spread of a **check mean over 38 examples**.
This probe has ten, so the observed spread must not be read as exp-01's — a
mean over ten examples is noisier by about √(38/10) ≈ 2.

Instead estimate the *per-example* between-replicate variance, which does not
depend on how many examples were run, and derive the check-level spread from
it:

$$\mathrm{SD}(\bar{s}_N) = \sqrt{\bar{v}/N}, \qquad \bar{v} = \text{mean per-example variance}$$

In [3]:
per_example_var = grid.var(axis=1, ddof=1)      # variance across replicates, per example
v_bar = per_example_var.mean()

print("per-example SD across replicates:")
print(np.sqrt(per_example_var).round(4).to_string())
print(f"\nmean per-example variance : {v_bar:.6f}")
print(f"mean per-example SD       : {np.sqrt(v_bar):.4f}")

per-example SD across replicates:
example
10.1038_emboj.2009.312        0.2582
10.1038_emboj.2009.340        0.0031
10.1038_embor.2009.217        0.0417
10.1038_embor.2009.233        0.0184
10.1038_s44318-025-00409-0    0.0312
10.1038_s44318-026-00715-1    0.0072
10.1038_s44319-025-00438-0    0.0338
10.1038_s44319-025-00631-1    0.0088
10.1038_s44320-025-00092-7    0.0183
10.1038_s44320-025-00094-5    0.0134

mean per-example variance : 0.007153
mean per-example SD       : 0.0846


### Does the extrapolation hold?

It assumes examples vary independently. Test it: the derived spread at N=10
should match the spread actually observed across the ten replicate means. If
they disagree materially, examples are correlated and the step to 38 is not
valid — say so in the note rather than using the number.

In [4]:
observed = grid.mean(axis=0).std(ddof=1)        # SD of the ten check means, N=10
derived_10 = np.sqrt(v_bar / grid.shape[0])

print(f"observed SD of check mean (N=10) : {observed:.4f}")
print(f"derived  SD of check mean (N=10) : {derived_10:.4f}")
ratio = observed / derived_10 if derived_10 else np.nan
print(f"ratio                            : {ratio:.2f}")
print("\n" + ("independence looks reasonable" if 0.7 <= ratio <= 1.4
               else "MISMATCH -- examples are not independent; do not extrapolate"))

observed SD of check mean (N=10) : 0.0273
derived  SD of check mean (N=10) : 0.0267
ratio                            : 1.02

independence looks reasonable


## What a replicate count buys

`SE(n) = SD(mean over 38) / √n`. Compare against the difference exp-01 is
looking for: if detailed-minus-minimal is smaller than `SE(5)`, five
replicates will not resolve it and the design needs changing rather than more
sampling.

In [5]:
sd_38 = np.sqrt(v_bar / FULL_BENCHMARK)
print(f"SD of a check mean over {FULL_BENCHMARK} examples: {sd_38:.4f}\n")
pd.DataFrame({
    "replicates": [1, 3, 5, 7, 10],
    "SE of one arm": [sd_38 / np.sqrt(n) for n in (1, 3, 5, 7, 10)],
    "SE of a paired difference (upper bound)":
        [sd_38 * np.sqrt(2) / np.sqrt(n) for n in (1, 3, 5, 7, 10)],
}).round(4)

SD of a check mean over 38 examples: 0.0137



,replicates,SE of one arm,SE of a paired difference (upper bound)
0,1,0.0137,0.0194
1,3,0.0079,0.0112
2,5,0.0061,0.0087
3,7,0.0052,0.0073
4,10,0.0043,0.0061


The second column is an **upper bound** on what exp-01 actually faces. Its
statistic is a difference of two arms, so if the arms varied independently
its variance would be twice one arm's. Pairing by example cancels whatever
noise is common to both, so the true value is lower — by how much, only exp-01
can say.

## Where the variance sits

If it is concentrated in one or two examples, that is a property of those
cases rather than of the model, and more replicates buy less than the average
suggests.

In [6]:
pd.DataFrame({
    "mean": grid.mean(axis=1),
    "sd": grid.std(axis=1, ddof=1),
    "min": grid.min(axis=1),
    "max": grid.max(axis=1),
}).sort_values("sd", ascending=False).round(3)

,mean,sd,min,max
example,,,,
10.1038_emboj.2009.312,0.728,0.258,0.000,0.837
10.1038_embor.2009.217,0.949,0.042,0.879,0.982
10.1038_s44319-025-00438-0,0.938,0.034,0.848,0.964
10.1038_s44318-025-00409-0,0.898,0.031,0.843,0.928
10.1038_embor.2009.233,0.905,0.018,0.886,0.933
10.1038_s44320-025-00092-7,0.903,0.018,0.879,0.939
10.1038_s44320-025-00094-5,0.894,0.013,0.870,0.914
10.1038_s44319-025-00631-1,0.905,0.009,0.893,0.921
10.1038_s44318-026-00715-1,0.981,0.007,0.968,0.988


## Non-response

A replicate that returned nothing scores as a fully missing row set rather
than being excluded. Even one changes how the average should be read.

In [7]:
import json as _json

empty = [
    {"replicate": int(rep.name.split("-")[1]),
     "example": str(p.parent.relative_to(rep))}
    for rep in reps
    for p in sorted(rep.rglob("prediction.json"))
    if not _json.loads(p.read_text())["outputs"]
]
pd.DataFrame(empty) if empty else "no empty answers in any replicate"

'no empty answers in any replicate'

## Cost

In [8]:
usage = [
    {"usd": u["total_cost_usd"], "turns": u.get("num_turns"),
     "seconds": (u.get("duration_ms") or 0) / 1000}
    for rep in reps
    for a in sorted(rep.rglob("tool_audit.json"))
    for u in [(_json.loads(a.read_text()).get("usage") or {})]
    if u.get("total_cost_usd") is not None
]
spend = pd.DataFrame(usage)
if len(spend):
    per_session = spend["usd"].mean()
    print(f"sessions    : {len(spend)}")
    print(f"per session : ${per_session:.4f}")
    print(f"this probe  : ${spend['usd'].sum():.2f}")
    print(f"median turns/seconds: {spend['turns'].median():.0f} / {spend['seconds'].median():.1f}")
    print("\nexp-01, 11 checks x 2 arms x 436 example-checks:")
    for n in (3, 5):
        print(f"  {n} replicates: {436*2*n:>5,} sessions  ~${436*2*n*per_session:,.0f}")
else:
    print("no usage recorded -- these runs predate cost capture")

sessions    : 100
per session : $0.0628
this probe  : $6.28
median turns/seconds: 4 / 32.6

exp-01, 11 checks x 2 arms x 436 example-checks:
  3 replicates: 2,616 sessions  ~$164
  5 replicates: 4,360 sessions  ~$274


## The decision goes in the note

Not here. Record which replicate count exp-01 will use, the `SE(n)` that
justified it, and whether the independence check passed — and keep the two
caveats visible: this is one check chosen for being noisy, and one arm rather
than a paired difference.